# Exploring Analysis Methods and Result Types

## Generate Constrained Models of the STM only condition and the Mixed STM condition

In [1]:
# Set the gurobi solver 

import os
from cobra.core.configuration import Configuration

os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

# Read in the gene weights CSV files

import pandas as pd

stm_threshold_10_90 = pd.read_csv("../imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("../imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("../imat_prep_test/stm_threshold_25_75.csv")

mixed_stm_threshold_10_90 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_10_90.csv")
mixed_stm_threshold_15_85 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_15_85.csv")
mixed_stm_threshold_25_75 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_25_75.csv")

# Make a python set of each gene weights CSV

stm_set_10_90 = set(stm_threshold_10_90["genes"].astype(str).str.strip())
stm_set_15_85 = set(stm_threshold_15_85["genes"].astype(str).str.strip())
stm_set_25_75 = set(stm_threshold_25_75["genes"].astype(str).str.strip())

mixed_stm_set_10_90 = set(mixed_stm_threshold_10_90["genes"].astype(str).str.strip())
mixed_stm_set_15_85 = set(mixed_stm_threshold_15_85["genes"].astype(str).str.strip())
mixed_stm_set_25_75 = set(mixed_stm_threshold_25_75["genes"].astype(str).str.strip())

# QC: Check the count of genes
print("STM1090 gene count",len(stm_set_10_90), "|", "STM1585 gene count", len(stm_set_15_85), "|", "STM2575 gene count", len(stm_set_25_75))
print("Mixed STM1090 gene count",len(mixed_stm_set_10_90), "|", "Mixed STM1585 gene count", len(mixed_stm_set_15_85), "|", "Mixed STM2575 gene count", len(mixed_stm_set_25_75))

# Load STM_V1_0

from cobra.io import load_model, load_json_model

control_model = load_model("STM_v1_0")

# Save all genes in the control model in a set

model_genes = set([g.id for g in control_model.genes])

# QC: Check the count of genes
print("STM_V1_0 Control Model has", len(model_genes), "total genes")

# Subset the gene weights lists for each sample to keep only the genes also found in the control model

stm1090_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm1585_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm2575_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

mixed_stm1090_model = mixed_stm_threshold_10_90[mixed_stm_threshold_10_90["genes"].isin(model_genes)]
mixed_stm1585_model = mixed_stm_threshold_15_85[mixed_stm_threshold_15_85["genes"].isin(model_genes)]
mixed_stm2575_model = mixed_stm_threshold_25_75[mixed_stm_threshold_25_75["genes"].isin(model_genes)]

# Align the gene weights lists to the control model genes list and assign missing genes a weight of 0

subset_weights_stm1090 = stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm1585 = stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm2575 = stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

subset_weights_mixedstm1090 = mixed_stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm1585 = mixed_stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm2575 = mixed_stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)


# QC: Confirm all samples have the same amount of gene weights

# 1. Make a list of the subset gene weights
subset_gene_weights = [
    subset_weights_stm1090,
    subset_weights_stm1585,
    subset_weights_stm2575,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
gene_counts = [len(weights) for weights in subset_gene_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(gene_counts)) == 1:
    print(f"All models have {gene_counts[0]} genes")
else:
    print("Gene counts differ between models")
    print(gene_counts)

# Convert gene weights to reaction weights with iMAT

from imatpy.parse_gpr import gene_to_rxn_weights

rxn_weights_stm1090 = gene_to_rxn_weights(control_model, subset_weights_stm1090)
rxn_weights_stm1585 = gene_to_rxn_weights(control_model, subset_weights_stm1585)
rxn_weights_stm2575 = gene_to_rxn_weights(control_model, subset_weights_stm2575)

rxn_weights_mixedstm1090 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1090)
rxn_weights_mixedstm1585 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1585)
rxn_weights_mixedstm2575 = gene_to_rxn_weights(control_model, subset_weights_mixedstm2575)

# QC: Confirm all samples have the same amount of reaction weights

# 1. Make a list of the reaction weights
reaction_weights = [
    rxn_weights_stm1090,
    rxn_weights_stm1585,
    rxn_weights_stm2575,
    rxn_weights_mixedstm1090,
    rxn_weights_mixedstm1585,
    rxn_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
reactions_counts = [len(weights) for weights in reaction_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(reactions_counts)) == 1:
    print(f"All models have {reactions_counts[0]} reactions")
else:
    print("Reaction counts differ between models")
    print(reactions_counts)

# Generate constrained models with iMAT by integrating the reaction weights

from imatpy.model_creation import generate_model

stm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1090, method = "imat_restrictions")
stm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1585, method = "imat_restrictions")
stm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm2575, method = "imat_restrictions")

mixedstm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1090, method = "imat_restrictions")
mixedstm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1585, method = "imat_restrictions")
mixedstm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm2575, method = "imat_restrictions")

STM1090 gene count 4671 | STM1585 gene count 4671 | STM2575 gene count 4671
Mixed STM1090 gene count 4671 | Mixed STM1585 gene count 4671 | Mixed STM2575 gene count 4671
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu
STM_V1_0 Control Model has 1271 total genes
All models have 1271 genes
All models have 2545 reactions
Read LP format model from file /tmp/tmpbewpdir8.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmp3vetxc2j.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpt532biov.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmp6s9skzf0.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmp9whse468.lp
Reading time 

## Dictionary of Models Generated

In [4]:
# make a dictionary of all the models 

models = {
    "stm_unconstrained": control_model,
    "mixed_stm1090": mixedstm1090_model,
    "mixed_stm1585": mixedstm1585_model,
    "mixed_stm2575": mixedstm2575_model,
    "stm_1090": stm1090_model,
    "stm_1585": stm1585_model,
    "stm_2575": stm2575_model
}

## Defining the Mouse Minimal Medium Conditions

What are the units for the value for each exchange reaction? How do we decide on a number when simulating for all exchange reactions? 
add tetrathionate to the minimal media and keep nitrate becuase it's mostly host derived. 

In [18]:
# Make a copy of the default medium from stm_v1_0
default_medium = dict(models["stm_unconstrained"].medium)

# Add mouse-derived, gut metabolites to the default medium to create a mouse gut minimal medium
mouse_min_medium = dict(default_medium)
mouse_min_medium.update({
    "EX_o2_e": 0.5, # updating to anoxic conditions
    "EX_no3_e": 0.5, # should we remove this?
    "EX_for_e": 4.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_glc__D_e": 0.0 # removing from default medium
})

## Optimize a model for growth using the minimal mouse media

In [19]:
test_model = models["mixed_stm1090"].copy()
mouse_medium = dict(mouse_min_medium)
test_model.medium = mouse_medium
solution = test_model.optimize()
summary = test_model.summary()

Read LP format model from file /tmp/tmp55pdnn5d.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros


## Optimized Model Summary

What do these results mean? What do they answer? What can we do with them? 

In [20]:
summary

Metabolite,Reaction,Flux,C-Number,C-Flux
arab__L_e,EX_arab__L_e,13.3,5,28.02%
asp__L_e,EX_asp__L_e,5,4,8.43%
ca2_e,EX_ca2_e,0.003651,0,0.00%
cl_e,EX_cl_e,0.003651,0,0.00%
cobalt2_e,EX_cobalt2_e,0.002434,0,0.00%
cu2_e,EX_cu2_e,0.002434,0,0.00%
fe2_e,EX_fe2_e,0.01673,0,0.00%
galct__D_e,EX_galct__D_e,10,6,25.28%
glcr_e,EX_glcr_e,10,6,25.28%
h2o_e,EX_h2o_e,8.488,0,0.00%


## Optimized Model Solution Flux

What do these results mean? What do they answer? What can we do with them? 

In [21]:
solution

,fluxes,reduced_costs
3HAD180,0.020000,NaN
3HAD181,0.020000,NaN
3HAD40,0.170280,NaN
3HAD60,0.170280,NaN
3HAD80,0.170280,NaN
...,...,...
12dgr2_ST,0.000072,NaN
OAL_ST,0.000722,NaN
OA4L_ST,0.000000,NaN
OA5L_ST,0.000722,NaN


## ATP summary for the optimized model 

In [ ]:
atp_summary = test_model.metabolites.atp_c.summary(solution=solution)

In [25]:
atp_summary

Percent,Flux,Reaction,Definition
53.35%,65.5,ACKr,ac_c + atp_c <=> actp_c + adp_c
0.01%,0.01,CYTK2,atp_c + dcmp_c <=> adp_c + dcdp_c
0.01%,0.01,DGK1,atp_c + dgmp_c <=> adp_c + dgdp_c
0.01%,0.01,NDPK5,atp_c + dgdp_c <=> adp_c + dgtp_c
0.03%,0.04014,NDPK6,atp_c + dudp_c <=> adp_c + dutp_c
0.01%,0.01,NDPK7,atp_c + dcdp_c <=> adp_c + dctp_c
16.20%,19.89,PGK,3pg_c + atp_c <=> 13dpg_c + adp_c
0.01%,0.01,PPAKr,adp_c + ppap_c <=> atp_c + ppa_c
30.35%,37.26,PYK,adp_c + h_c + pep_c --> atp_c + pyr_c
0.03%,0.04014,URIDK2r,atp_c + dump_c <=> adp_c + dudp_c


## NADH Summary for optimized model

In [28]:
nadh_summary = test_model.metabolites.nadh_c.summary(solution=solution)
nadh_summary

Percent,Flux,Reaction,Definition
0.07%,0.02,ACALD,acald_c + coa_c + nad_c <=> accoa_c + h_c + nadh_c
0.04%,0.01,AKGDH,akg_c + coa_c + nad_c --> co2_c + nadh_c + succoa_c
0.04%,0.01,ALCD19,glyald_c + h_c + nadh_c <=> glyc_c + nad_c
0.04%,0.01079,DKMPPD3,dkmpp_c + 3.0 h2o_c + 2.0 nad_c --> 2kmb_c + for_c + 4.0 h_c + 2.0 nadh_c + pi_c
71.48%,19.89,GAPD,g3p_c + nad_c + pi_c <=> 13dpg_c + h_c + nadh_c
0.14%,0.03853,GCALDD,gcald_c + h2o_c + nad_c --> glyclt_c + 2.0 h_c + nadh_c
0.04%,0.01,GLYCDx,glyc_c + nad_c --> dha_c + h_c + nadh_c
0.11%,0.03162,HACD8i,3hodcoa_c + nad_c --> 3ohodcoa_c + h_c + nadh_c
0.44%,0.1235,HISTD,h2o_c + histd_c + 2.0 nad_c --> 3.0 h_c + his__L_c + 2.0 nadh_c
0.73%,0.2042,IMPD,h2o_c + imp_c + nad_c --> h_c + nadh_c + xmp_c


## Summaries for all metabolites in the minimal mouse media of the optimized model

In [42]:
# copy the medium of the test model
mouse_medium = test_model.medium.copy()

# make a list of the keys in the medium of the test model 
rxn_ids = list(mouse_medium.keys())

# make a list of reaction objects of the reactions in the medium of the test model
rxn_obj_list = []

for rxn in rxn_ids:
    rxn_obj = test_model.reactions.get_by_id(rxn)
    rxn_obj_list.append(rxn_obj)

print(rxn_obj_list)

[<Reaction EX_arab__L_e at 0x11651e795750>, <Reaction EX_asp__L_e at 0x11651e797e50>, <Reaction EX_ca2_e at 0x11651e7a4ad0>, <Reaction EX_cbl1_e at 0x11651e7a4e10>, <Reaction EX_cl_e at 0x11651e7a6dd0>, <Reaction EX_co2_e at 0x11651e7a70d0>, <Reaction EX_cobalt2_e at 0x11651e7a73d0>, <Reaction EX_cu2_e at 0x11651efbc810>, <Reaction EX_mg2_e at 0x11651e7c2f10>, <Reaction EX_fe2_e at 0x11651e7c3990>, <Reaction EX_fe3_e at 0x11651e7c3b10>, <Reaction EX_mn2_e at 0x11651e7c3f90>, <Reaction EX_mobd_e at 0x11651e7d0450>, <Reaction EX_na1_e at 0x11651e7d1050>, <Reaction EX_nh4_e at 0x11651e7d1ad0>, <Reaction EX_for_e at 0x11651e7d1dd0>, <Reaction EX_no3_e at 0x11651e7d26d0>, <Reaction EX_o2_e at 0x11651e7d2b50>, <Reaction EX_galct__D_e at 0x11651e7e0b10>, <Reaction EX_pi_e at 0x11651e7e1110>, <Reaction EX_zn2_e at 0x11651e7e2190>, <Reaction EX_pyr_e at 0x11651e7e39d0>, <Reaction EX_so4_e at 0x11651e7f5ed0>, <Reaction EX_glcr_e at 0x11651e605350>, <Reaction EX_tungs_e at 0x11651e6071d0>, <React

In [ ]:
# get the first metabolite of the reaction
metabolite = next(iter(rxn_obj_list[0].metabolites))

# generate the metabolite summary
metabolite.summary(solution=solution)

Percent,Flux,Reaction,Definition
100.00%,13.3,EX_arab__L_e,arab__L_e <=>
Percent,Flux,Reaction,Definition
100.00%,-13.3,ARBtex,arab__L_e <=> arab__L_p


## Simulating with FVA

In [46]:
from cobra.flux_analysis import flux_variability_analysis

test_fva = test_model.copy()

flux_variability_analysis(test_fva, test_model.reactions[:10], fraction_of_optimum=0.9)


Read LP format model from file /tmp/tmptxs3c2hf.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros


,minimum,maximum
3HAD180,0.000000,0.207261
3HAD181,0.004526,0.167474
3HAD40,0.153252,0.601338
3HAD60,0.153252,0.601338
3HAD80,0.153252,0.601338
3KGK,0.000000,0.000000
3NTD2pp,0.000000,0.000000
3NTD4pp,0.000000,0.000000
3NTD7pp,0.000000,0.000000
3NTD9pp,0.000000,0.000000


In [ ]:
test_fva2 = test_model.copy()

test_fva2.optimize()
test_fva2.summary(fva=0.95)

Read LP format model from file /tmp/tmpmyugmvy9.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
